# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

As this assignment utilizes global variables in the skeleton code, to avoid having errors in your code you can either:

1. You can place all of your code within the function definitions for all of the questions (other than import statements).
2. You can create copies of all the global variables with the copy() method and proceed as usual.

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [ ]:

import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def clean_st(string):
    if isinstance(string, str):
        return re.sub(r'\[.+?\]', '', string).strip()
    return string

def get_cities(sport_name):
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    cities.columns = ['Metropolitan area', 'Population', 'NFL', 'MLB', 'NBA', 'NHL']
    cities['Population'] = pd.to_numeric(cities['Population'], errors='coerce')
    cities[sport_name] = cities[sport_name].apply(clean_st)
    cities_sport = cities[['Metropolitan area', sport_name, 'Population']].copy()
    cities_sport = cities_sport[cities_sport[sport_name].str.len() > 1]
    cities_sport = cities_sport[cities_sport[sport_name] != '—']
    return cities_sport

def get_sport_wl(sport_name):
    cities_sport = get_cities(sport_name)
    df = pd.read_csv(f"assets/{sport_name.lower()}.csv")
    df = df[df['year'] == 2018]
    if sport_name == 'NHL':
        df = df[~df['W'].str.contains('Division', na=False)]
    if sport_name == 'NFL':
        df = df[~df['W'].str.contains('AFC', na=False)]
        df = df[~df['W'].str.contains('NFC', na=False)]
        
    df['team'] = df['team'].apply(lambda x: re.sub(r'[\*+]|\(.+\)', '', x).strip())
    df['W'] = pd.to_numeric(df['W'], errors='coerce')
    df['L'] = pd.to_numeric(df['L'], errors='coerce')
    df = df.dropna(subset=['W', 'L'])
    
    mapping = {}
    for idx, row in df.iterrows():
        team = row['team']
        if team == "Toronto Maple Leafs": last_word = "Maple Leafs"
        elif team == "Detroit Red Wings": last_word = "Red Wings"
        elif team == "Columbus Blue Jackets": last_word = "Blue Jackets"
        elif team == "Vegas Golden Knights": last_word = "Golden Knights"
        elif team == "Portland Trail Blazers": last_word = "Trail Blazers"
        elif team == "Boston Red Sox": last_word = "Red Sox"
        elif team == "Chicago White Sox": last_word = "White Sox"
        elif team == "Toronto Blue Jays": last_word = "Blue Jays"
        elif team == "New York Knicks": last_word = "Knicks"
        elif team == "New York Rangers": last_word = "Rangers"
        elif team == "New York Islanders": last_word = "Islanders"
        elif team == "San Francisco 49ers": last_word = "49ers"
        else:
            last_word = team.split(' ')[-1]
        
        for c_idx, c_row in cities_sport.iterrows():
            if last_word in c_row[sport_name]:
                mapping[team] = c_row['Metropolitan area']
                break
                
    df['Metropolitan area'] = df['team'].map(mapping)
    df = df.dropna(subset=['Metropolitan area'])
    df['W/L'] = df['W'] / (df['W'] + df['L'])
    avg_wl = df.groupby('Metropolitan area')['W/L'].mean().reset_index()
    return pd.merge(cities_sport[['Metropolitan area', 'Population']], avg_wl, on='Metropolitan area', how='inner')

def nhl_correlation():

    res = get_sport_wl('NHL')
    population_by_region = res['Population'].tolist()
    win_loss_by_region = res['W/L'].tolist()
    
    assert len(population_by_region) == len(win_loss_by_region), "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q1: There should be 28 teams being analysed for NHL"
    
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [ ]:
def nba_correlation():

    res = get_sport_wl('NBA')
    population_by_region = res['Population'].tolist()
    win_loss_by_region = res['W/L'].tolist()
    assert len(population_by_region) == len(win_loss_by_region), "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q2: There should be 28 teams being analysed for NBA"
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [ ]:
def mlb_correlation():

    res = get_sport_wl('MLB')
    population_by_region = res['Population'].tolist()
    win_loss_by_region = res['W/L'].tolist()
    assert len(population_by_region) == len(win_loss_by_region), "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, "Q3: There should be 26 teams being analysed for MLB"
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [ ]:
def nfl_correlation():

    res = get_sport_wl('NFL')
    population_by_region = res['Population'].tolist()
    win_loss_by_region = res['W/L'].tolist()
    assert len(population_by_region) == len(win_loss_by_region), "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, "Q4: There should be 29 teams being analysed for NFL"
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [ ]:
def sports_team_performance():

    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)
    
    wl_dfs = {s: get_sport_wl(s) for s in sports}
    
    for s1 in sports:
        for s2 in sports:
            if s1 == s2:
                continue
            merged = pd.merge(wl_dfs[s1], wl_dfs[s2], on='Metropolitan area', how='inner')
            pval = stats.ttest_rel(merged['W/L_x'], merged['W/L_y'])[1]
            p_values.loc[s1, s2] = pval
    return p_values
